# Advanced Transfer Learning — EfficientNet-B4 + ViT-B/16

This notebook trains two state-of-the-art models on HAM10000:

| Model | Architecture | Input | Pretrain | Strength |
|-------|-------------|-------|----------|----------|
| EfficientNet-B4 | Compound-scaled CNN | 380×380 | ImageNet | Accuracy/efficiency tradeoff |
| ViT-B/16 | Transformer | 224×224 | ImageNet21k | Global context, long-range patterns |

**Advanced techniques:**
- Two-stage training (backbone freeze → gradual unfreeze)
- Discriminative learning rates (lower LR for backbone)
- Focal Loss with class weights
- MixUp + CutMix augmentation
- Cosine annealing with warm restarts
- AMP (mixed precision training)
- Temperature scaling (post-hoc calibration)

In [2]:
import sys, os
sys.path.insert(0, '..')
os.makedirs('../checkpoints', exist_ok=True)
os.makedirs('../results', exist_ok=True)

import torch
import torch.optim as optim
import json
import numpy as np

from src.dataset import build_dataloaders, compute_class_weights, CLASS_NAMES
from src.models  import build_model, get_param_groups
from src.losses  import build_loss
from src.train   import train, build_scheduler, evaluate, TemperatureScaler
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_roc_curves, plot_training_history

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM: 17.2 GB


## 1. Load Data

In [3]:
# For EfficientNet-B4 use larger input (380); for ViT use 224
IMAGE_SIZE   = 380    # change to 224 for ViT
BATCH_SIZE   = 16     # reduce to 8 if GPU OOM with 380px
DATA_ROOT    = '../dataset'

train_loader, val_loader, test_loader, test_df = build_dataloaders(
    data_root        = DATA_ROOT,
    image_size       = IMAGE_SIZE,
    batch_size       = BATCH_SIZE,
    use_oversampling = True,
    use_mixup_cutmix = True,
    use_hair_aug     = True,
)

# Compute class weights for loss function
import pandas as pd
meta_df = pd.read_csv(f'{DATA_ROOT}/HAM10000_metadata.csv')
labels_all = [CLASS_NAMES.index(x) for x in meta_df['dx']]
class_weights = compute_class_weights(labels_all).to(DEVICE)
print('Class weights:', class_weights.cpu().numpy().round(3))

[Dataset] Train: 7054 | Val: 1464 | Test: 1497
[Dataset] Class dist (train): {'nv': np.int64(4730), 'mel': np.int64(777), 'bkl': np.int64(775), 'bcc': np.int64(365), 'akiec': np.int64(233), 'vasc': np.int64(98), 'df': np.int64(76)}
Class weights: [0.943 0.6   0.281 2.682 0.277 0.046 2.172]


## 2. Train EfficientNet-B4

In [4]:
# ── Build model ────────────────────────────────────────────────────────────
effnet = build_model('efficientnet_b4', num_classes=7, pretrained=True)
effnet = effnet.to(DEVICE)

total_params    = sum(p.numel() for p in effnet.parameters())
trainable_params = sum(p.numel() for p in effnet.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to C:\Users\ghosh/.cache\torch\hub\checkpoints\efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:07<00:00, 11.0MB/s]


Total params:     18,471,247
Trainable params: 18,471,247


In [5]:
# ── Loss & optimizer ────────────────────────────────────────────────────────
criterion = build_loss('focal', class_weights=class_weights, gamma=2.0,
                        label_smoothing=0.1)

# Discriminative LRs: head=1e-4, backbone=1e-5
param_groups = get_param_groups(effnet, base_lr=1e-4, backbone_lr_multiplier=0.1)
optimizer    = optim.AdamW(param_groups, weight_decay=1e-4)

scheduler = build_scheduler(optimizer, warmup_epochs=3, total_epochs=40,
                             T_0=10, T_mult=2, eta_min=1e-6)

print('Criterion:', criterion)
print('Optimizer:', optimizer)

Criterion: FocalLoss()
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 1.0000000000000002e-06
    maximize: False
    weight_decay: 0.0001

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 1e-05
    lr: 1.0000000000000001e-07
    maximize: False
    weight_decay: 0.0001
)


In [6]:
# ── Train ────────────────────────────────────────────────────────────────────
history_effnet = train(
    model           = effnet,
    train_loader    = train_loader,
    val_loader      = val_loader,
    criterion       = criterion,
    optimizer       = optimizer,
    scheduler       = scheduler,
    device          = DEVICE,
    num_epochs      = 40,
    freeze_epochs   = 3,
    patience        = 10,
    checkpoint_dir  = '../checkpoints',
    model_name      = 'efficientnet_b4',
    grad_accum_steps = 2,
)

# Save history
with open('../results/history_efficientnet_b4.json', 'w') as f:
    # Convert numpy floats for JSON serialization
    def convert(obj):
        if isinstance(obj, (np.float32, np.float64)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj
    import json
    history_save = {
        k: [{kk: float(vv) for kk, vv in ep.items() if not isinstance(vv, np.ndarray)}
            for ep in v]
        for k, v in history_effnet.items()
    }
    json.dump(history_save, f, indent=2)

[Train] Freezing backbone for 3 epochs...


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:201: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 001/40 | LR: 3.40e-05 | Train Loss: 1.4992  Acc: 0.1419  BalAcc: 0.1424 | Val Loss: 0.3205  Acc: 0.0813  BalAcc: 0.1473 | 66.7s
  ✓ New best | BalAcc: 0.1473 → saved to ../checkpoints\efficientnet_b4_best.pth


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 002/40 | LR: 6.70e-05 | Train Loss: 1.2968  Acc: 0.2763  BalAcc: 0.2758 | Val Loss: 0.2078  Acc: 0.5143  BalAcc: 0.5205 | 59.8s
  ✓ New best | BalAcc: 0.5205 → saved to ../checkpoints\efficientnet_b4_best.pth


...

Epoch 039/40 | LR: 9.46e-05 | Train Loss: 0.5961  Acc: 0.6510  BalAcc: 0.6510 | Val Loss: 0.0889  Acc: 0.7650  BalAcc: 0.7573 | 114.3s


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 040/40 | LR: 9.27e-05 | Train Loss: 0.6086  Acc: 0.6570  BalAcc: 0.6565 | Val Loss: 0.0864  Acc: 0.7698  BalAcc: 0.7693 | 115.6s
  ✓ New best | BalAcc: 0.7693 → saved to ../checkpoints\efficientnet_b4_best.pth


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:269: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


[Train] Loaded best checkpoint (epoch 40, BalAcc: 0.7693)


In [7]:
plot_training_history(history_effnet, '../results/training_efficientnet_b4.png')

[Eval] Saved training curves → ../results/training_efficientnet_b4.png


## 3. Evaluate EfficientNet-B4 on Test Set

In [8]:
# Reload best checkpoint
ckpt = torch.load('../checkpoints/efficientnet_b4_best.pth', map_location=DEVICE)
effnet.load_state_dict(ckpt['model_state_dict'])

# Apply temperature scaling
ts = TemperatureScaler()
ts.fit(effnet, val_loader, DEVICE)

# Evaluate
test_criterion = build_loss('label_smoothing', class_weights=class_weights)
test_metrics = evaluate(effnet, test_loader, test_criterion, DEVICE)

metrics = evaluate_model(
    test_metrics['all_probs'],
    test_metrics['all_preds'],
    test_metrics['all_targets'],
    bootstrap_ci=True,
    save_dir='../results'
)

plot_confusion_matrix(test_metrics['all_targets'], test_metrics['all_preds'],
                      '../results/cm_efficientnet_b4.png')
plot_roc_curves(test_metrics['all_probs'], test_metrics['all_targets'],
                '../results/roc_efficientnet_b4.png')

C:\Users\ghosh\AppData\Local\Temp\ipykernel_25240\2031477138.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load('../checkpoints/efficientnet_b4_best.pth',

[TemperatureScaler] Optimal T = -32.9062


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Balanced Accuracy: 0.7270 (95% CI: 0.6834–0.7695)

  Accuracy:          0.7381
  Balanced Accuracy: 0.7270
  Cohen's Kappa:     0.7405
  AUC (macro OvR):   0.9560
  AUC (weighted):    0.9452
              precision    recall  f1-score   support

       akiec       0.40      0.69      0.50        51
         bcc       0.69      0.77      0.73        77
         bkl       0.60      0.68      0.64       157
          df       0.43      0.59      0.50        22
         mel       0.40      0.71      0.51       168
          nv       0.96      0.75      0.85      1000
        vasc       0.56      0.91      0.69        22

    accuracy                           0.74      1497
   macro avg       0.58      0.73      0.63      1497
weighted avg       0.81      0.74      0.76      1497

[Eval] Saved confusion matrix → ../results/cm_efficientnet_b4.png
[Eval] Saved ROC curves → ../results/roc_efficientnet_b4.png


## 4. Train Vision Transformer (ViT-B/16)

In [9]:
# ViT requires 224×224 input — reload data with correct size
train_loader_224, val_loader_224, test_loader_224, _ = build_dataloaders(
    data_root=DATA_ROOT, image_size=224, batch_size=32,
    use_oversampling=True, use_mixup_cutmix=True,
)

vit = build_model('vit_b16', num_classes=7, pretrained=True).to(DEVICE)
trainable = sum(p.numel() for p in vit.parameters() if p.requires_grad)
print(f'ViT-B/16 trainable params: {trainable:,}')

criterion_vit = build_loss('label_smoothing', class_weights=class_weights,
                            label_smoothing=0.1)

# ViT benefits from lower LR and AdamW
param_groups_vit = get_param_groups(vit, base_lr=5e-5, backbone_lr_multiplier=0.1)
optimizer_vit = optim.AdamW(param_groups_vit, weight_decay=0.05)
scheduler_vit = build_scheduler(optimizer_vit, warmup_epochs=5, total_epochs=40,
                                 T_0=10, T_mult=2, eta_min=1e-7)

[Dataset] Train: 7054 | Val: 1464 | Test: 1497
[Dataset] Class dist (train): {'nv': np.int64(4730), 'mel': np.int64(777), 'bkl': np.int64(775), 'bcc': np.int64(365), 'akiec': np.int64(233), 'vasc': np.int64(98), 'df': np.int64(76)}


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\ghosh/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:31<00:00, 11.1MB/s] 


ViT-B/16 trainable params: 86,196,999


In [10]:
history_vit = train(
    model=vit, train_loader=train_loader_224, val_loader=val_loader_224,
    criterion=criterion_vit, optimizer=optimizer_vit, scheduler=scheduler_vit,
    device=DEVICE, num_epochs=40, freeze_epochs=5, patience=10,
    checkpoint_dir='../checkpoints', model_name='vit_b16',
    grad_accum_steps=2,
)
plot_training_history(history_vit, '../results/training_vit_b16.png')

[Train] Freezing backbone for 5 epochs...


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:201: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 001/40 | LR: 1.04e-05 | Train Loss: 2.0282  Acc: 0.1497  BalAcc: 0.1493 | Val Loss: 0.6051  Acc: 0.0806  BalAcc: 0.1499 | 64.7s
  ✓ New best | BalAcc: 0.1499 → saved to ../checkpoints\vit_b16_best.pth


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 002/40 | LR: 2.03e-05 | Train Loss: 1.8307  Acc: 0.1933  BalAcc: 0.1949 | Val Loss: 0.5595  Acc: 0.0533  BalAcc: 0.2946 | 65.4s
  ✓ New best | BalAcc: 0.2946 → saved to ../checkpoints\vit_b16_best.pth


...

Epoch 039/40 | LR: 4.88e-05 | Train Loss: 0.7488  Acc: 0.6440  BalAcc: 0.6402 | Val Loss: 0.3975  Acc: 0.4570  BalAcc: 0.7137 | 72.6s
  ✓ New best | BalAcc: 0.7137 → saved to ../checkpoints\vit_b16_best.pth


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 040/40 | LR: 4.81e-05 | Train Loss: 0.7491  Acc: 0.6499  BalAcc: 0.6437 | Val Loss: 0.3931  Acc: 0.3429  BalAcc: 0.7005 | 72.6s


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:269: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


[Train] Loaded best checkpoint (epoch 39, BalAcc: 0.7137)
[Eval] Saved training curves → ../results/training_vit_b16.png


In [11]:
# Evaluate ViT
ckpt_vit = torch.load('../checkpoints/vit_b16_best.pth', map_location=DEVICE)
vit.load_state_dict(ckpt_vit['model_state_dict'])

test_metrics_vit = evaluate(vit, test_loader_224, criterion_vit, DEVICE)
metrics_vit = evaluate_model(
    test_metrics_vit['all_probs'], test_metrics_vit['all_preds'],
    test_metrics_vit['all_targets'], bootstrap_ci=True, save_dir='../results'
)
plot_confusion_matrix(test_metrics_vit['all_targets'], test_metrics_vit['all_preds'],
                      '../results/cm_vit_b16.png')
plot_roc_curves(test_metrics_vit['all_probs'], test_metrics_vit['all_targets'],
                '../results/roc_vit_b16.png')

C:\Users\ghosh\AppData\Local\Temp\ipykernel_25240\1989657133.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt_vit = torch.load('../checkpoints/vit_b16_best.pth', map

Balanced Accuracy: 0.7018 (95% CI: 0.6689–0.7330)

  Accuracy:          0.4422
  Balanced Accuracy: 0.7018
  Cohen's Kappa:     0.4673
  AUC (macro OvR):   0.9233
  AUC (weighted):    0.9196
              precision    recall  f1-score   support

       akiec       0.21      0.86      0.33        51
         bcc       0.44      0.79      0.56        77
         bkl       0.57      0.40      0.47       157
          df       0.13      0.82      0.23        22
         mel       0.29      0.70      0.41       168
          nv       1.00      0.34      0.50      1000
        vasc       0.15      1.00      0.25        22

    accuracy                           0.44      1497
   macro avg       0.40      0.70      0.39      1497
weighted avg       0.79      0.44      0.48      1497

[Eval] Saved confusion matrix → ../results/cm_vit_b16.png
[Eval] Saved ROC curves → ../results/roc_vit_b16.png


## 5. Results Summary

Collect and compare model performance.

In [12]:
import pandas as pd

results = pd.DataFrame([
    {
        'Model':              'EfficientNet-B4',
        'Balanced Acc':       metrics.get('balanced_accuracy', 0),
        'AUC (macro)':        metrics.get('auc_macro', 0),
        "Cohen's Kappa":      metrics.get('cohen_kappa', 0),
        'Accuracy':           metrics.get('accuracy', 0),
    },
    {
        'Model':              'ViT-B/16',
        'Balanced Acc':       metrics_vit.get('balanced_accuracy', 0),
        'AUC (macro)':        metrics_vit.get('auc_macro', 0),
        "Cohen's Kappa":      metrics_vit.get('cohen_kappa', 0),
        'Accuracy':           metrics_vit.get('accuracy', 0),
    },
])
print(results.round(4).to_string(index=False))

          Model  Balanced Acc  AUC (macro)  Cohen's Kappa  Accuracy
EfficientNet-B4        0.7270       0.9560         0.7405    0.7381
       ViT-B/16        0.7018       0.9233         0.4673    0.4422
